Cell 1 — Load data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
ADVANCED_DIR = BASE_DIR / "data" / "advanced"

train = pd.read_csv(PROCESSED_DIR / "train_features.csv")
train["date"] = pd.to_datetime(train["date"])

print(train.shape)

(3000888, 21)


Cell 2 — Sort data

In [2]:
train = train.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

print(train.shape)
train.head()

(3000888, 21)


,id,date,store_nbr,family,sales,onpromotion,year,month,day,quarter,...,day_of_week,is_weekend,city,state,type,cluster,dcoilwtico,is_payday,is_national_holiday,earthquake_period
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,...,1,0,Quito,Pichincha,D,13,93.14,0,1,0
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2013,1,2,1,...,2,0,Quito,Pichincha,D,13,93.14,0,0,0
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,2013,1,3,1,...,3,0,Quito,Pichincha,D,13,92.97,0,0,0
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,2013,1,4,1,...,4,0,Quito,Pichincha,D,13,93.12,0,0,0
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,2013,1,5,1,...,5,1,Quito,Pichincha,D,13,93.12,0,1,0


Cell 3 — Expanded lag features

In [3]:
lag_days = [1, 2, 3, 7, 14, 21, 28, 56]

for lag in lag_days:
    train[f"sales_lag_{lag}"] = (
        train.groupby(["store_nbr", "family"])["sales"]
        .shift(lag)
    )

print("Lag features created.")

Lag features created.


Cell 4 — Expanded rolling mean features

In [4]:
rolling_windows = [3, 7, 14, 28, 56]

for window in rolling_windows:
    train[f"rolling_mean_{window}"] = (
        train.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

print("Rolling mean features created.")

Rolling mean features created.


Cell 5 — Rolling median, min, max

In [5]:
for window in [7, 14, 28]:
    train[f"rolling_median_{window}"] = (
        train.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).median())
    )

    train[f"rolling_min_{window}"] = (
        train.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).min())
    )

    train[f"rolling_max_{window}"] = (
        train.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).max())
    )

print("Rolling median/min/max features created.")

Rolling median/min/max features created.


Cell 6 — Rolling standard deviation

In [6]:
for window in [7, 14, 28, 56]:
    train[f"rolling_std_{window}"] = (
        train.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

print("Rolling std features created.")

Rolling std features created.


Cell 7 — Promotion rolling features

In [7]:
for window in [7, 14, 28]:
    train[f"promo_rolling_mean_{window}"] = (
        train.groupby(["store_nbr", "family"])["onpromotion"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

print("Promotion rolling features created.")

Promotion rolling features created.


Cell 8 — Fill missing values

In [8]:
new_features = [
    col for col in train.columns
    if "lag_" in col
    or "rolling_" in col
]

train[new_features] = train[new_features].fillna(0)

print(train[new_features].isnull().sum().sum())
print(train.shape)

0
(3000888, 50)


Cell 9 — Save expanded dataset

In [9]:
train.to_csv(
    ADVANCED_DIR / "train_expanded_features.csv",
    index=False
)

print("Saved train_expanded_features.csv")
print(train.shape)

Saved train_expanded_features.csv
(3000888, 50)
